### Given a sim, plot the number of reactions used and kinetic target scatter

In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast, Any
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from yaml import emit

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [38]:
def plot_rxn_usage(
        time_num: int,
        fba: dict,
        metabolism: object,
        output: dict,
        show_plot: bool = True,
        save_plot: bool = True,
        save_path: Optional[str] = None,
        renderer: Optional[str] = None,
        new_genes: bool = False,
):

    # --- Load Data ---
    counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

    reaction_names = metabolism.reaction_names
    sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names).mul(counts_to_molar, axis=0)
    if new_genes:
        new_fba_rxn_ids = metabolism.parameters['fba_new_reaction_ids']
        sim_flux = sim_flux[new_fba_rxn_ids].copy()
        plot_name = f'new_genes_reaction_usage_pie_chart.svg'
        reaction_names = new_fba_rxn_ids.copy()
    else:
        plot_name = f'reaction_usage_pie_chart.svg'

    # --- Report Minimum Flux ---
    min_reaction_flux = sim_flux.replace(0, np.nan).min()
    min_sim_flux = min_reaction_flux.min()
    print(f'The all time non-zero minimum flux across all reactions is {min_sim_flux:0.3e} mmol/L/s')

    # --- Prepare Data for Plot ---
    # calculate the number of reactions used in the simulation
    non_zero_flux = sim_flux.gt(min_sim_flux, axis=0).sum()
    num_reactions_always_used = np.sum(non_zero_flux == time_num)
    num_reactions_never_used = np.sum(non_zero_flux == 0)
    num_reactions_occasionally_used = np.sum((non_zero_flux > 0) & (non_zero_flux < time_num))

    assert num_reactions_always_used + num_reactions_never_used + num_reactions_occasionally_used == len(reaction_names)

    # plot pie chart for distribution of reactions
    df_plot = pd.DataFrame({
        'Usage': ['Always Used', 'Never Used', 'Occasionally Used'],
        'Count': [num_reactions_always_used, num_reactions_never_used, num_reactions_occasionally_used]
    })
    fig = px.pie(df_plot, values='Count', names='Usage',
                 title=f'Distribution of Reaction Usage in Simulation - {experiment_name}',
                 color_discrete_sequence=px.colors.qualitative.Pastel,
                 labels={'Usage': 'Reaction Usage', 'Count': 'Number of Reactions'})
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.update_layout(showlegend=False,
                      template="plotly_white",
                      paper_bgcolor='rgba(255, 0, 0, 0)',
                      plot_bgcolor='rgba(255, 0, 0, 0)',
                      title_font_size=20,
                      )

    # --- Show and/or Save Plot ---
    if show_plot:
        fig.show(renderer=renderer)

    if save_plot:

        assert save_path is not None, "Please provide a save path to save the plot."

        save_path = f'{save_path}figures/'

        if not os.path.exists(save_path):
            os.makedirs(save_path)
            print(f"Directory '{save_path}' created.")

        fig.write_image(f'{save_path}{plot_name}', width=600, height=600, scale=3)
    return

In [35]:
from sklearn.metrics import r2_score
def plot_kinetic_scatter(
        fba: dict,
        metabolism: object,
        output: dict,
        in_molar:bool = True,
        show_plot: bool = True,
        save_plot: bool = True,
        save_path: Optional[str] = None,
        renderer: Optional[str] = None,
        new_genes: bool = False,
):
    # --- Load Data ---
    counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

    reaction_names = metabolism.reaction_names
    kinetic_reactions = metabolism.kinetic_constraint_reactions
    if in_molar:
        sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names).mul(counts_to_molar, axis=0)
        kinetic_target = pd.DataFrame(fba["target_kinetic_fluxes"], columns=kinetic_reactions).mul(counts_to_molar, axis=0)
        unit_label = 'mmol/L/hr'
    else:
        sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names)
        kinetic_target = pd.DataFrame(fba["target_kinetic_fluxes"], columns=kinetic_reactions)
        unit_label = 'counts/hr'

    if new_genes:
        new_fba_rxn_ids = metabolism.parameters['fba_new_reaction_ids']
        sim_flux = sim_flux[new_fba_rxn_ids].copy()
        plot_name = f'new_genes_kinetic_target_scatter.svg'
        reaction_names = new_fba_rxn_ids.copy()
    else:
        plot_name = f'kinetic_target_scatter.svg'

    # --- Process Data for Plot ---
    sim_flux_mean = sim_flux.mean(axis=0).to_frame(name="mean_flux") * 3600 + 1E-6 # mmol/L/hr or counts/hr
    sim_flux_mean = np.log10(sim_flux_mean)
    sim_flux_mean = sim_flux_mean.replace(-np.inf, 0)

    kinetic_target_mean = kinetic_target.mean(axis=0) * 3600 + 1E-6 # mmol/L/hr or counts/hr
    kinetic_target_mean = np.log10(kinetic_target_mean)
    kinetic_target_mean = kinetic_target_mean.replace(-np.inf, 0)

    sim_flux_mean["kinetic"] = [kinetic_target_mean[r] if r in kinetic_reactions else False for r in reaction_names]

    # --- Plot Scatter ---
    # plot scatter of kinetic target vs mean flux for kinetic reactions
    kinetic_reactions_flux = sim_flux_mean[sim_flux_mean['kinetic'] != False]

    fig = px.scatter(kinetic_reactions_flux, x='kinetic', y='mean_flux',
                     title='Mean Flux vs Kinetic Target for Kinetic Reactions',
                     labels={'kinetic': f'Mean Kinetic Target ({unit_label})', 'mean_flux': f'Mean Estimated Flux ({unit_label})'},
                     hover_data=['kinetic', 'mean_flux', kinetic_reactions_flux.index],
                     color_discrete_sequence=px.colors.qualitative.Pastel)
    # add y=x line
    fig.add_trace(go.Scatter(x=[-6, kinetic_reactions_flux['kinetic'].max()], y=[-6, kinetic_reactions_flux['kinetic'].max()],
                              mode='lines', name='y=x', line=dict(color=px.colors.qualitative.Pastel[1], dash='dash')))

    # calculate R^2 value for logged kinetic target vs mean flux
    r2 = r2_score(kinetic_reactions_flux['kinetic'], kinetic_reactions_flux['mean_flux'])
    fig.add_annotation(x=0.05, y=0.95, xref='paper', yref='paper',
                       text=f'R² = {r2:.2f}', showarrow=False, font=dict(size=14))
    fig.update_layout(template="plotly_white",
                       showlegend=False,
                       paper_bgcolor='rgba(255, 0, 0, 0)',
                       plot_bgcolor='rgba(255, 0, 0, 0)',
                       title_font_size=20,
                       )

    # --- Show and/or Save Plot ---
    if show_plot:
        fig.show(renderer=renderer)

    if save_plot:

        assert save_path is not None, "Please provide a save path to save the plot."

        save_path = f'{folder}figures/'

        if not os.path.exists(save_path):
            os.makedirs(save_path)
            print(f"Directory '{save_path}' created.")

        fig.write_image(f'{save_path}{plot_name}', width=800, height=600, scale=3)

    return

In [28]:
def format_number(x):
    """Format number: scientific notation if |x| < 0.01, else 1 decimal place"""
    if abs(x) < 0.01 and x != 0:
        return f'{x:.2E}'
    else:
        return f'{x:.1f}'

In [42]:
import plotly.graph_objects as go
import numpy as np

def plot_flux_distribution(
        fba: dict,
        metabolism: object,
        output: dict,
        in_molar: bool = True,
        show_plot: bool = True,
        save_plot: bool = True,
        save_path: Optional[str] = None,
        renderer: Optional[str] = None,
        new_genes: bool = False,
):
    # --- Load Data ---
    counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

    reaction_names = metabolism.reaction_names
    if in_molar:
        sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names).mul(counts_to_molar, axis=0)
        unit_label = 'mmol/L/s'
    else:
        sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names)
        unit_label = 'counts/s'

    if new_genes:
        new_fba_rxn_ids = metabolism.parameters['fba_new_reaction_ids']
        sim_flux = sim_flux[new_fba_rxn_ids].copy()
        plot_name = f'new_genes_mean_flux_distribution.svg'
    else:
        plot_name = f'mean_flux_distribution.svg'
    # --- Process Data for Plot ---
    sim_flux_mean = sim_flux.mean(axis=0)

    # --- Define Custom Bins ---
    if in_molar:
        custom_bins = [0, np.min(counts_to_molar), 1, max(sim_flux.max(axis=0))/2, max(sim_flux.max(axis=0))]
    else:
        custom_bins = [0, 1, 2, max(sim_flux.max(axis=0))/10, max(sim_flux.max(axis=0))/2, max(sim_flux.max(axis=0))]

    binned_data, bin_edges = np.histogram(sim_flux_mean, bins=custom_bins)

    # Create bar chart with custom bins
    fig = go.Figure()

    bin_centers = [(custom_bins[i] + custom_bins[i+1]) / 2 for i in range(len(custom_bins)-1)]
    bin_widths = [custom_bins[i+1] - custom_bins[i] for i in range(len(custom_bins)-1)]
    bin_labels = [f'[{format_number(custom_bins[i])}, {format_number(custom_bins[i+1])})'
                  for i in range(len(custom_bins)-1)]

    fig.add_trace(go.Bar(
        x=bin_labels,
        y=binned_data,
        marker=dict(
            color=px.colors.qualitative.Pastel[0],
            line=dict(color='white', width=1)
        ),
        text=binned_data,
        textposition='auto',
    ))

    fig.update_layout(
        template="plotly_white",
        showlegend=False,
        paper_bgcolor='rgba(255, 0, 0, 0)',
        plot_bgcolor='rgba(255, 0, 0, 0)',
        title='Distribution of Mean Fluxes Across Reactions',
        title_font_size=20,
        xaxis_title=f'Mean Estimated Flux ({unit_label})',
        yaxis_title='Count',
    )

    # --- Show and/or Save Plot ---
    if show_plot:
        fig.show(renderer=renderer)

    if save_plot:
        assert save_path is not None, "Please provide a save path to save the plot."

        save_path = f'{save_path}figures/'

        if not os.path.exists(save_path):
            os.makedirs(save_path)
            print(f"Directory '{save_path}' created.")

        fig.write_image(f'{save_path}{plot_name}', width=600, height=600, scale=3)

In [7]:
# import sim
time_num = 2973
date = '2026-05-18'
experiment_name = 'baseline'
condition = 'basal'
experiment_type = 'Fig1'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [8]:
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

In [36]:
folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
# plot_flux_distribution(fba, metabolism, output, in_molar=False, show_plot = True, renderer = 'browser', save_plot = True, save_path = folder)
# plot_rxn_usage(fba, metabolism, output, show_plot = True, renderer = 'browser', save_plot = True, save_path = folder)
plot_kinetic_scatter(fba, metabolism, output, in_molar=True, show_plot = True, renderer = 'browser', save_plot = True, save_path = folder, new_genes=True)

In [39]:
folder = f'out/{experiment_type}/{condition}/{experiment_name}_{time_num}_{date}/'
plot_rxn_usage(time_num, fba, metabolism, output, show_plot = True, renderer = 'browser', save_plot = True, save_path = folder, new_genes=True)
print(f'Min counts to molar: {min(counts_to_molar)}')

The all time non-zero minimum flux across all reactions is 6.984e-07 mmol/L/s
Min counts to molar: 6.98445387567962e-07


In [43]:
plot_flux_distribution(fba, metabolism, output, in_molar=True, show_plot = True, renderer = 'browser', save_plot = True, save_path = folder, new_genes=True)